## Score - 	0.84562
- #### updated feature engineering by removing unnecessary features
- #### added climate features.

In [35]:
import numpy as np 
import pandas as pd  
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [36]:
pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/SampleSubmission.csv').sample(5)

,ID,TargetF1,TargetRAUC
977,ID_C42E904C,0,0
837,ID_D411550E,0,0
682,ID_5F839CA2,0,0
4,ID_0E02825D,0,0
489,ID_CB5F44E3,0,0


In [37]:
data_dict =pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/data_dictionary.csv')

In [38]:
df = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Train.csv')

In [39]:
df2 = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/climate_features.csv')
df=df.merge(df2,on='ID')

In [40]:
df["is_climate_sensitive"].value_counts()

is_climate_sensitive
1    2047
0    1099
Name: count, dtype: int64

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3146 entries, 0 to 3145
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    3146 non-null   object 
 1   zone                  3146 non-null   object 
 2   gender                3146 non-null   object 
 3   deathdate_x           3146 non-null   object 
 4   age                   3146 non-null   float64
 5   avg_temperature       3146 non-null   float64
 6   max_temperature       3146 non-null   float64
 7   min_temperature       3146 non-null   float64
 8   precipitation         3146 non-null   float64
 9   latitude              3146 non-null   float64
 10  longitude             3146 non-null   float64
 11  location              3146 non-null   object 
 12  is_climate_sensitive  3146 non-null   int64  
 13  deathdate_y           3146 non-null   object 
 14  elevation             3146 non-null   int64  
 15  hot_days_30d         

In [42]:
def preprocessor(data, fit=True, target=None, te_maps=None):
    data = data.copy()
    data.rename(columns={"deathdate_x": "deathdate"}, inplace=True)

    data["zone"] = (data["zone"] == "Peri_urban").astype(int)
    data["gender"] = (data["gender"] == "Male").astype(int)
    data["age"] = data["age"].astype(int)
    data["age_group"] = pd.cut(data["age"], bins=[-1,10,20,30,40,50,60,70,80,90,np.inf], labels=False) + 1
    data["is_young"] = (data["age"] <= 20).astype(int)
    data["is_old"] = (data["age"] >= 70).astype(int)
    data["age_zero"] = (data["age"] == 0).astype(int)
    data["is_rain"] = (data["precipitation"] > 0).astype(int)

    data["precip_log"] = np.log1p(data["precipitation"])
    data["precip_per_temp"] = data["precipitation"] / (abs(data["avg_temperature"]) + 1)
    data["temp_rain_interaction"] = data["avg_temperature"] * data["precipitation"]
    data["temp_range"] = data["max_temperature"] - data["min_temperature"]
    data["avg_temp_position"] = (data["avg_temperature"] - data["min_temperature"]) / (data["max_temperature"] - data["min_temperature"] + 1e-6)

    data["latitude"] = (data["latitude"] - data["latitude"].mean()) / data["latitude"].std()
    data["longitude"] = (data["longitude"] - data["longitude"].mean()) / data["longitude"].std()
    data["lat_gt_long"] = (data["latitude"] > data["longitude"]).astype(int)
    data["deathdate"] = pd.to_datetime(data["deathdate"], errors="coerce")
    data["date"] = data["deathdate"].dt.day
    data["month"] = data["deathdate"].dt.month
    data["year"] = data["deathdate"].dt.year
    data["day_of_week"] = data["deathdate"].dt.dayofweek
    data["day_of_year"] = data["deathdate"].dt.dayofyear
    data["week_of_year"] = data["deathdate"].dt.isocalendar().week.astype(int)

    data.drop(columns=["hotdays_30"], errors="ignore", inplace=True)

    te_cols = ["zone", "gender", "age_group", "location"]

    if fit:
        if target is None:
            raise ValueError("target must be provided when fit=True")
        global_mean = data[target].mean()
        te_maps = {}
        for col in te_cols:
            stats = pd.DataFrame({"feature": data[col], "target": data[target]}).groupby("feature")["target"].agg(["mean", "count"])
            smoothing = 10
            stats["encoded"] = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)
            te_maps[col] = {"mapping": stats["encoded"].to_dict(), "global_mean": global_mean}
            data[f"{col}_te"] = data[col].map(te_maps[col]["mapping"]).fillna(global_mean)
    else:
        if te_maps is None:
            raise ValueError("te_maps must be provided when fit=False")
        for col in te_cols:
            data[f"{col}_te"] = data[col].map(te_maps[col]["mapping"]).fillna(te_maps[col]["global_mean"])

    data["zone_gender"] = data["zone_te"] * data["gender_te"]
    data["latitude_gender"] = data["latitude"] * data["gender_te"]
    data["longitude_gender"] = data["longitude"] * data["gender_te"]
    data["longitude_zone"] = data["longitude"] * data["zone_te"]

    data.drop(columns=["deathdate", "deathdate_x", "deathdate_y", "zone_te", "location"], errors="ignore", inplace=True)

    if fit:
        return data, te_maps
    return data

In [43]:
df1, enc_dict = preprocessor(df, fit=True, target='is_climate_sensitive')

In [44]:
X = df1.drop(columns=["is_climate_sensitive", "ID", ])
y = df1["is_climate_sensitive"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [45]:
models = {

    "Gradient Boosting": GradientBoostingClassifier(
        learning_rate=0.01,
        min_samples_leaf=10,
        min_samples_split=15,
        n_estimators=300,
        random_state=42,
        subsample=0.9,
        max_depth=3
    ),

    "Extra Trees": ExtraTreesClassifier(
        max_depth=5,
        max_features=None,
        min_samples_leaf=2,
        min_samples_split=5,
        n_estimators=500,
        n_jobs=-1,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        max_depth=5,
        max_features=None,
        min_samples_split=10,
        n_estimators=700,
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        colsample_bytree=1.0,
        eval_metric="logloss",
        gamma=0.1,
        learning_rate=0.01,
        max_depth=4,
        min_child_weight=3,
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        learning_rate=0.01,
        n_estimators=300,

        max_depth=4,
        num_leaves=15,

        min_child_samples=10,
        min_split_gain=0.1,

        subsample=0.9,
        colsample_bytree=1.0,

        reg_alpha=0.0,
        reg_lambda=0.0,

        objective="binary",
        verbosity=-1,
        random_state=42,
        n_jobs=-1
    ),

    "CatBoost": CatBoostClassifier(
        learning_rate=0.01,
        iterations=300,

        depth=4,

        l2_leaf_reg=3,
        min_data_in_leaf=10,

        random_strength=1.0,
        rsm=1.0,

        loss_function="Logloss",
        eval_metric="AUC",

        random_seed=42,
        verbose=False,
        thread_count=-1
    )
}
tuned_models = {}
results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    tuned_models[name] = model

    train_prob = model.predict_proba(X_train)[:, 1]
    train_pred = (train_prob >= 0.5).astype(int)

    train_f1 = f1_score(y_train, train_pred)
    train_auc = roc_auc_score(y_train, train_prob)
    train_score = 0.6 * train_f1 + 0.4 * train_auc

    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)

    test_f1 = f1_score(y_test, test_pred)
    test_auc = roc_auc_score(y_test, test_prob)
    test_score = 0.6 * test_f1 + 0.4 * test_auc

    results.append({
        "Model": name,

        "Train F1": train_f1,
        "Train ROC-AUC": train_auc,
        "Train Final Score": train_score,

        "Test F1": test_f1,
        "Test ROC-AUC": test_auc,
        "Test Final Score": test_score,

        "Parameters": model.get_params()
    })

results_df = pd.DataFrame(results)

In [46]:
results_df = pd.DataFrame(results)

results_df["Train-Test Final Score Difference"] = (
    results_df["Train Final Score"] - results_df["Test Final Score"]
).abs()

results_df = results_df.sort_values(
    by=[
        "Train-Test Final Score Difference",
        "Test Final Score"
    ],
    ascending=[
        True,
        False
    ]
).reset_index(drop=True)

results_df

,Model,Train F1,Train ROC-AUC,Train Final Score,Test F1,Test ROC-AUC,Test Final Score,Parameters,Train-Test Final Score Difference
0,CatBoost,0.820483,0.847468,0.831277,0.825688,0.829191,0.827089,"{'iterations': 300, 'learning_rate': 0.01, 'de...",0.004187
1,Extra Trees,0.825930,0.867220,0.842446,0.818824,0.837190,0.826170,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_...",0.016276
2,Gradient Boosting,0.844549,0.870713,0.855014,0.823529,0.837955,0.829299,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'...",0.025715
3,XGBoost,0.865778,0.898220,0.878755,0.812573,0.831292,0.820060,"{'objective': 'binary:logistic', 'base_score':...",0.058695
4,Random Forest,0.875189,0.901854,0.885855,0.812721,0.831774,0.820342,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.065513
5,LightGBM,0.870030,0.903240,0.883314,0.809802,0.824933,0.815854,"{'boosting_type': 'gbdt', 'class_weight': None...",0.067459


In [47]:
tf = pd.read_csv('/kaggle/input/datasets/vedikagupta0/climate-risk-health-prediction-challenge/Test.csv')

In [51]:
tf=tf.merge(df2,on='ID')

In [55]:
tf1 = preprocessor(tf, fit=False, te_maps=enc_dict)

In [56]:
tf1.head()

,ID,zone,gender,age,avg_temperature,max_temperature,min_temperature,precipitation,latitude,longitude,...,day_of_week,day_of_year,week_of_year,gender_te,age_group_te,location_te,zone_gender,latitude_gender,longitude_gender,longitude_zone
0,ID_E760D84B,0,0,75,21.224387,24.632729,18.145633,4.295335,0.569921,0.591260,...,5,328,47,0.671414,0.297386,0.650668,0.440893,0.382653,0.396980,0.388259
1,ID_6EDEA907,0,0,50,21.708596,25.184528,18.285274,1.376196,0.834305,0.226250,...,2,23,4,0.671414,0.497098,0.650668,0.440893,0.560164,0.151908,0.148570
2,ID_B9FFC8D8,0,0,76,21.371149,24.584523,18.866490,2.336643,0.537167,0.629812,...,4,74,11,0.671414,0.297386,0.650668,0.440893,0.360661,0.422865,0.413574
3,ID_74C6C94E,1,0,90,21.341990,25.188662,17.998317,4.703071,-2.364024,-2.541572,...,2,107,16,0.671414,0.302667,0.650668,0.430128,-1.587239,-1.706448,-1.628209
4,ID_0E02825D,0,1,8,19.710391,22.905167,18.082628,4.066863,0.537167,0.629812,...,0,112,17,0.631726,0.828350,0.650668,0.414831,0.339342,0.397869,0.413574


In [57]:
def make_submission(
    test_data,
    model,
    id_col="ID"
):
    ids = test_data.pop(id_col)

    pred_binary = model.predict(test_data)
    pred_prob = model.predict_proba(test_data)[:, 1]
    submission = pd.DataFrame({
        "ID": ids,
        "TargetF1": pred_binary.astype(int),
        "TargetRAUC": pred_prob
    })

    return submission

In [59]:
best_name = results_df.iloc[0]["Model"]
best_model = tuned_models[best_name]

X_full = df1.drop(columns=["is_climate_sensitive", "ID"])
y_full = df1["is_climate_sensitive"]

best_model.fit(X_full, y_full)
X_submission = tf1.drop(columns=["ID"])
ids = tf1["ID"].copy()

pred_prob = best_model.predict_proba(X_submission)[:, 1]
pred_binary = (pred_prob >= 0.5).astype(int)

submission = pd.DataFrame({
    "ID": ids,
    "TargetF1": pred_binary,
    "TargetRAUC": pred_prob
})

submission.to_csv("climate-risk-catboost-climate.csv", index=False)

print("Best model:", best_name)
print(submission.head())

Best model: CatBoost
            ID  TargetF1  TargetRAUC
0  ID_E760D84B         0    0.384401
1  ID_6EDEA907         1    0.560530
2  ID_B9FFC8D8         0    0.406623
3  ID_74C6C94E         0    0.403683
4  ID_0E02825D         1    0.851718
